In [11]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [9]:
os.getcwd()

'/workspaces/IC-RNA-2025/fuction_ensemble_1/redes-ensemble-s/Teste02 copy'

In [12]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [16]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            try:
                df = read_excel(file)
                if "r2" not in df.columns:
                    print(f"[Ignorado] {file} não contém a coluna 'r2'")
                    continue
                architecture = read_txt(file.replace(".xlsx", ".txt"))
                self.concatenate_df(df, architecture)
            except Exception as e:
                print(f"[Erro] Falha ao processar {file}: {e}")
        if not self.results_df.empty:
            self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)
        else:
            print("[Aviso] Nenhum DataFrame válido foi carregado.")


    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [17]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



[Aviso] Nenhum DataFrame válido foi carregado.


KeyError: 'r2'

In [15]:
from pandas import read_excel

results = get_files(subfolder="results", extension="xlsx")

for file in results:
    df = read_excel(file)
    print(f"Arquivo: {file}")
    print("Colunas disponíveis:", df.columns.tolist())
    print("=" * 40)


Arquivo: /workspaces/IC-RNA-2025/fuction_ensemble_1/redes-ensemble-s/Teste02 copy/content/results/metrics_2_8.xlsx
Colunas disponíveis: ['Unnamed: 0', 'r2', 'r2_sup', 'r2_test', 'r2_val', 'r2_vt', 'mse', 'mse_sup', 'mse_test', 'mse_val', 'mse_vt', 'mape', 'rmse', 'r2_adj', 'rsd', 'aic', 'bic']
Arquivo: /workspaces/IC-RNA-2025/fuction_ensemble_1/redes-ensemble-s/Teste02 copy/content/results/metrics_2_3.xlsx
Colunas disponíveis: ['Unnamed: 0', 'r2', 'r2_sup', 'r2_test', 'r2_val', 'r2_vt', 'mse', 'mse_sup', 'mse_test', 'mse_val', 'mse_vt', 'mape', 'rmse', 'r2_adj', 'rsd', 'aic', 'bic']
Arquivo: /workspaces/IC-RNA-2025/fuction_ensemble_1/redes-ensemble-s/Teste02 copy/content/results/metrics_2_1.xlsx
Colunas disponíveis: ['Unnamed: 0', 'r2', 'r2_sup', 'r2_test', 'r2_val', 'r2_vt', 'mse', 'mse_sup', 'mse_test', 'mse_val', 'mse_vt', 'mape', 'rmse', 'r2_adj', 'rsd', 'aic', 'bic']
Arquivo: /workspaces/IC-RNA-2025/fuction_ensemble_1/redes-ensemble-s/Teste02 copy/content/results/metrics_2_5.xlsx
